# VOZ-HSD - Proposed XLM-RoBERTa + ViCLSR

Notebook Kaggle chạy hai model trong nhóm proposed `FacebookAI/xlm-roberta-base` và `huynhtin/ViCLSR`.

Bật Internet và GPU T4 trước khi chạy. ViCLSR dùng XLM-RoBERTa-Large nên chậm và tốn VRAM hơn XLM-RoBERTa-base.

In [1]:
import os
import subprocess
from pathlib import Path

EXP_REL = Path("notebooks/models/proposed/VOZ-HSD - Proposed XLM-RoBERTa_ViCLSR")
ROOT = Path.cwd()

# If this notebook is opened outside the repo, clone the project first.
if not (ROOT / EXP_REL / "run_two_models.py").exists():
    if not (ROOT / "ViAmpleHate").exists():
        subprocess.run(["git", "clone", "-b", "trung-dev", "https://github.com/MinhTuan2405/ViAmpleHate.git"], check=True)
    os.chdir(ROOT / "ViAmpleHate")

ROOT = Path.cwd()
EXP_DIR = ROOT / EXP_REL
SCRIPT = str(EXP_DIR / "run_two_models.py")
REQS = str(EXP_DIR / "requirements.txt")

assert Path(SCRIPT).exists(), f"Không tìm thấy {SCRIPT}"
assert Path(REQS).exists(), f"Không tìm thấy {REQS}"
print("Repo path:", ROOT)
print("Proposed path:", EXP_DIR)


Repo path: /kaggle/working/ViAmpleHate
Proposed path: /kaggle/working/ViAmpleHate/notebooks/models/proposed/VOZ-HSD - Proposed XLM-RoBERTa_ViCLSR


In [2]:
!pip install -q -r "{REQS}"

## Cấu hình

Notebook này cố định chạy `VOZ-HSD`. Theo protocol phân phối tự nhiên: lấy mẫu phân tầng 100.000 dòng, giữ tỉ lệ lớp tự nhiên và chia train/dev/test 80/10/10.

Hai model dùng cùng dataset, split, seed và max length. Training budget được ghi rõ: XLM-RoBERTa-base chạy 5 epochs; ViCLSR chạy 2 epochs với FP16 và batch size 4 vì backbone XLM-RoBERTa-Large nặng hơn đáng kể trên Kaggle. Đây là thí nghiệm proposed theo giới hạn tài nguyên, không phải compute-matched.


In [3]:
DATASET = "vozhsd"
XLMR_EPOCHS = 5
VICLSR_EPOCHS = 2
MAX_LEN = 128
OUTPUT_DIR = "/kaggle/working/viamplehate_runs_vozhsd_seed42"

# Giữ protocol phân phối tự nhiên đã dùng trong completed run.
VOZ_SPLIT_POLICY = "baseline"
VOZ_SAMPLE_SIZE = 100_000
VOZ_HATE_RATIO = 0.10  # Chỉ được dùng khi policy="proposed".


## 1. Chạy XLM-RoBERTa

Cấu hình full run: 5 epochs, batch size 8, max length 128.


In [4]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model xlm-roberta \
  --epochs {XLMR_EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 8 \
  --eval-batch-size 16 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR}


Device: cuda
Dataset: vozhsd
Model: xlm-roberta -> FacebookAI/xlm-roberta-base
VOZ-HSD policy=baseline | sampled=100,000/10,747,733 | hate_ratio=0.0544 | split=(0.8, 0.1, 0.1)
train: 80,000 rows | labels={0: 75646, 1: 4354}
val: 10,000 rows | labels={0: 9430, 1: 570}
test: 10,000 rows | labels={0: 9486, 1: 514}
Tokenizing dataset...
Loading model weights...
Model ready. Starting training...
epoch=1 train_loss=0.7739 val_acc=0.9593 val_macro_f1=0.7985 val_hate_f1=0.6186
saved best checkpoint -> /kaggle/working/viamplehate_runs_vozhsd_seed42/vozhsd/xlm-roberta/best_model.pt
epoch=2 train_loss=0.5056 val_acc=0.9600 val_macro_f1=0.8303 val_hate_f1=0.6820
saved best checkpoint -> /kaggle/working/viamplehate_runs_vozhsd_seed42/vozhsd/xlm-roberta/best_model.pt
epoch=3 train_loss=0.3771 val_acc=0.9684 val_macro_f1=0.8285 val_hate_f1=0.6736
epoch=4 train_loss=0.2842 val_acc=0.9682 val_macro_f1=0.8501 val_hate_f1=0.7171
saved best checkpoint -> /kaggle/working/viamplehate_runs_vozhsd_seed42/vozh

## 2. Chạy ViCLSR

Cấu hình full run: 2 epochs, FP16, batch size 4, max length 128. ViCLSR dùng XLM-RoBERTa-Large nên training budget thấp hơn XLM-RoBERTa-base để phù hợp giới hạn thời gian/VRAM Kaggle.


In [5]:
!python "{SCRIPT}" \
  --dataset {DATASET} \
  --model viclsr \
  --epochs {VICLSR_EPOCHS} \
  --max-len {MAX_LEN} \
  --batch-size 4 \
  --eval-batch-size 8 \
  --voz-split-policy {VOZ_SPLIT_POLICY} \
  --voz-sample-size {VOZ_SAMPLE_SIZE} \
  --voz-hate-ratio {VOZ_HATE_RATIO} \
  --output-dir {OUTPUT_DIR} \
  --fp16


Device: cuda
Dataset: vozhsd
Model: viclsr -> huynhtin/ViCLSR
VOZ-HSD policy=baseline | sampled=100,000/10,747,733 | hate_ratio=0.0544 | split=(0.8, 0.1, 0.1)
train: 80,000 rows | labels={0: 75646, 1: 4354}
val: 10,000 rows | labels={0: 9430, 1: 570}
test: 10,000 rows | labels={0: 9486, 1: 514}
Tokenizing dataset...
Loading model weights...
Some weights of XLMRobertaModel were not initialized from the model checkpoint at huynhtin/ViCLSR and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
Loaded ViCLSR projection head: mlp.dense.weight/bias
Model ready. Starting training...
epoch=1 train_loss=0.6956 val_acc=0.9430 val_macro_f1=0.4853 val_hate_f1=0.0000
saved best checkpoint -> /kaggle/working/viamplehate_runs_vozhsd_seed42/vozhsd/viclsr/best_model.pt
epoch=2 train_loss=0.6918 val_acc=0.9430 val_macro_f1=0.4853 val_hate_f1=0.0000

TEST
{
  "accuracy": 0.9486,
  "macro_f1": 0.48681104382633683,
  "hate_f1": 0.0
}
              precision    recall  f1-score   support

 

## 3. Xem metrics

Mỗi model lưu `best_model.pt`, `metrics.json`, và tokenizer vào `/kaggle/working`.

In [6]:
import json
from pathlib import Path

base = Path(OUTPUT_DIR) / DATASET
for model_name in ["xlm-roberta", "viclsr"]:
    path = base / model_name / "metrics.json"
    print("\n===", model_name, "===")
    if not path.exists():
        print("Chưa có metrics:", path)
        continue
    metrics = json.loads(path.read_text(encoding="utf-8"))
    print({k: metrics[k] for k in ["accuracy", "macro_f1", "hate_f1"]})
    print(metrics["report"])


=== xlm-roberta ===
{'accuracy': 0.9706, 'macro_f1': 0.8458471861134111, 'hate_f1': 0.7071713147410359}
              precision    recall  f1-score   support

    NON-HATE     0.9833    0.9858    0.9845      9486
        HATE     0.7245    0.6907    0.7072       514

    accuracy                         0.9706     10000
   macro avg     0.8539    0.8382    0.8458     10000
weighted avg     0.9700    0.9706    0.9703     10000


=== viclsr ===
{'accuracy': 0.9486, 'macro_f1': 0.48681104382633683, 'hate_f1': 0.0}
              precision    recall  f1-score   support

    NON-HATE     0.9486    1.0000    0.9736      9486
        HATE     0.0000    0.0000    0.0000       514

    accuracy                         0.9486     10000
   macro avg     0.4743    0.5000    0.4868     10000
weighted avg     0.8998    0.9486    0.9236     10000


## Ghi chú báo cáo

Ưu tiên so sánh `macro_f1` và `hate_f1` vì class `HATE` ít hơn nhiều so với `NON-HATE`. Báo cáo phải nêu XLM-RoBERTa chạy 5 epochs và ViCLSR chạy 2 epochs; không mô tả đây là so sánh cùng training budget. File `metrics.json` lưu cấu hình thực tế trong trường `config`.


## Kết quả full run đã lưu

| Model | Epochs | Batch | FP16 | Accuracy | Macro-F1 | HATE-F1 |
|---|---:|---:|:---:|---:|---:|---:|
| XLM-RoBERTa-base | 5 | 8 | No | 0.9706 | 0.8458 | 0.7072 |
| ViCLSR | 2 | 4 | Yes | 0.9486 | 0.4868 | 0.0000 |

XLM-RoBERTa học được lớp HATE, trong khi ViCLSR collapse về lớp NON-HATE trên split mất cân bằng này. Outputs trong notebook được lưu từ hai Kaggle processes hoàn tất (`returncode=0`). Các checkpoint dung lượng lớn được loại khỏi Git và lightweight artifacts được archive trong `output/`.
